# MIMIC-IV GPU Training

Train BiLSTM, BiGRU, and Transformer models on Google Colab GPU. When Colab asks for Google Drive access, choose `yeminenisaiswaroopa@gmail.com`. Before running, set `PROJECT_DIR` to the Google Drive folder that contains this repository and `data/mimic_iv_raw/mimic-iv-2-1.zip`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

# Change this if your folder name/location is different in Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/Temporal Analysis')
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())
print('MIMIC-IV ZIP exists:', (PROJECT_DIR / 'data/mimic_iv_raw/mimic-iv-2-1.zip').exists())

In [ ]:
!nvidia-smi
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Full Preprocess + GPU Training

This cell rebuilds the complete preprocessed arrays from `mimic-iv-2-1.zip`, trains BiLSTM, BiGRU, and Transformer on GPU, then writes metrics, checkpoints, and plots back into this Drive folder.

In [ ]:
!python src/run_full_colab_training.py --zip data/mimic_iv_raw/mimic-iv-2-1.zip --chunk 100000 --seq_len 6 --epochs 80 --lr 0.0003 --batch_size 128 --hidden_size 128 --patience 12

## Inspect Final Arrays

Use this after the full pipeline cell finishes to confirm the split sizes and positive class rate.

In [ ]:
import numpy as np

for split in ['train', 'val', 'test']:
    X = np.load(f'data/preprocessed/X_{split}.npy')
    y = np.load(f'data/preprocessed/y_{split}.npy')
    print(split, X.shape, y.shape, 'positive_rate=', round(float(y.mean()), 4), 'finite=', bool(np.isfinite(X).all()))

In [ ]:
!python evaluate_trained_models.py

In [ ]:
# Optional plots after all checkpoints exist.
!python src/reporting/plot_model_results.py